# Suite VAL — Validators

`Validators.Validate(store, predicates)` checks each predicate on its matches among the objects given: every combination
of them whose schemas fit its symbols. It reports what does not hold, what is unknown and what raised, each match
labelled as mbse-schemas labels objects.

In [ ]:
from mbse.Expressions import Expressions as E
from mbse.Expressions.Dialects.Python import Text
from mbse.Patterns import Constraints as C, Distributions as D, Predicates as P, Validators as V
from support import Contact, Directory, Phone, book, listed, raises

the, c, p = E.variable("the"), E.variable("c"), E.variable("p")
owns = E.quantifier("any", "e", E.operation("entries", c, "phones"), E.variable("e").phone.eq(p))
predicates = C.OfSet.Builder().predicates(
    lambda b: b.name("Adult").symbols({"the": Contact}).requires(the.age.ge(18)),
    lambda b: b.name("Phoned").symbols({"the": Contact}).requires(E.operation("count", E.operation("entries", the, "phones")).ge(1)),
    lambda b: b.name("OwnedNumbered").symbols({"c": Contact, "p": Phone}).requires(E.operation("implies", owns, p.has("number"))),
).create()
store = book()
ann = store.Contact().name("ann").age(30).phones(lambda e: e.phone(lambda q: q.number("555"))).create()
bob = store.Contact().name("bob").phones(lambda e: e.phone(lambda q: q)).create()

## VAL-01 · A predicate is checked on each match

In [ ]:
validate = V.Validate(store, predicates)
assert validate(Contact, ann) == []
assert validate(Contact, bob) == ["the=Contact#0: 'Adult' is unknown"]  # bob's age is absent; he alone has no phone match
young = store.Contact().name("cy").age(9).create()
assert validate(Contact, young) == ["the=Contact#0: 'Adult' does not hold", "the=Contact#0: 'Phoned' does not hold"]
assert validate(Contact, store.Phone().create()) == ["the value is a 'Phone', not an instance of the given schema"]
assert validate(Phone, store.Phone().create()) == []  # no predicate has a match of a phone alone

## VAL-02 · Reachable checks every match among what the root reaches: the cross product of the symbols

In [ ]:
listed(store, ann, bob)
problems = V.Validate(store, predicates).Reachable(Directory, store.singleton("book.Directory"))
assert problems == ["the=Contact#2: 'Adult' is unknown",
                    "c=Contact#2, p=Phone#4: 'OwnedNumbered' does not hold"], problems  # bob's phone has no number
assert V.Validate(store, predicates).Reachable(Contact, ann) == ["the=Contact#3: 'Adult' is unknown",
    "c=Contact#3, p=Phone#4: 'OwnedNumbered' does not hold"]  # from ann, through the directory, bob is reached too

## VAL-03 · An unknown result is reported, ignored or a violation

In [ ]:
assert V.Validate(store, predicates, unknown="ignore")(Contact, bob) == []
assert V.Validate(store, predicates, unknown="violation")(Contact, bob) == ["the=Contact#0: 'Adult' does not hold"]
assert V.Validate(store, predicates, unknown="report")(Contact, bob) == ["the=Contact#0: 'Adult' is unknown"]
with raises(ValueError, match="unknown must be 'report', 'ignore' or 'violation', got 'maybe'"):
    V.Validate(store, predicates, unknown="maybe")
assert V.UNKNOWN == ("report", "ignore", "violation")

## VAL-04 · A rule that raises is reported in place; a predicate that cannot be right is refused

In [ ]:
raising = C.OfSet.Builder().predicates(
    lambda b: b.name("Mixed").symbols({"the": Contact}).requires(E.operation("add", the.age, 1.5).ge(2.0)),
    lambda b: b.name("Number").symbols({"the": Contact}).requires(the.age)).create()
assert V.Validate(store, raising)(Contact, ann) == [
    "the=Contact#0: 'Mixed' raised TypeError: add expects numbers of one domain, got int and float",
    "the=Contact#0: 'Number' raised TypeError: a predicate must be a bool, got int"]
with raises(ValueError, match="predicate 'Older' has parameters: it is checked where it is applied"):
    V.Validate(store, [lambda b: b.name("Older").symbols({"the": Contact}).parameters(lambda p: p.name("n")).requires(
        the.age.gt(E.variable("n")))])
with raises(ValueError, match="predicate 'X': rule: variable 'n' is not bound"):
    V.Validate(store, [lambda b: b.name("X").symbols({"the": Contact}).requires(E.variable("n"))])

## VAL-05 · A pattern is validated as any predicate is

In [ ]:
person = E.variable("person")
HasName = (P.OfPredicate.Builder().name("HasName").symbols({"person": Contact}).parameters(lambda p: p.name("name"))
           .requires(Text.FromFunction(lambda person, name: person.name == name)).create())
Ages = P.OfPredicate.Builder().name("Ages").symbols({"person": Contact}).requires(D.Choices.Builder().arms(
    lambda a: a.weight(3).requires(HasName(person, "senior"), person.age.ge(65), D.Normal.Builder().symbol("age").mean(70)
                                   .deviation(8).rounded().requires(lambda person, age: person.age == age).create()),
    lambda a: a.weight(7).requires(HasName(person, "adult"), D.Uniform.Builder().symbol("age").low(18).high(64)
                                   .requires(lambda person, age: person.age == age).create()),
).decreasing().create()).create()
people = book()
listed(people, *(people.Contact().name(n).age(a).create() for n, a in (("senior", 70), ("senior", 40), ("adult", 30))),
       people.Contact().name("adult").create(), people.Contact().name("kid").age(9).create())
assert V.Validate(people, [Ages]).Reachable(Directory, people.singleton("book.Directory")) == [
    "person=Contact#2: 'Ages' does not hold",  # a senior under 65: no arm holds
    "person=Contact#4: 'Ages' is unknown",  # an adult without an age: the uniform has no witness
    "person=Contact#5: 'Ages' does not hold"]  # a kid falls under no arm
assert V.Validate(people, [Ages], unknown="ignore").Reachable(Directory, people.singleton("book.Directory")) == [
    "person=Contact#2: 'Ages' does not hold", "person=Contact#5: 'Ages' does not hold"]